# Market Sentiment Model
In this notebook we are going to explore building a transformer model to analyze overall market sentiment based off of current
news articles.

In [7]:
import kagglehub
import numpy as np
import pandas as pd
from torch import nn
from torch.optim import lr_scheduler
from torch.utils.data import TensorDataset, DataLoader

from src.my_engine.config import ModelConfig, TrainerConfig
from src.my_engine.trainer import Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm

In [8]:
# Download latest version
path = kagglehub.dataset_download("sbhatti/financial-sentiment-analysis")

print("Path to dataset files:", path)

df = pd.read_csv(f"{path}/data.csv")
df['Sentiment'] = df['Sentiment'].astype('category')
df['Sentiment'] = df['Sentiment'].cat.codes
print(df.shape)
df.head()

Path to dataset files: /home/alexsearle/.cache/kagglehub/datasets/sbhatti/financial-sentiment-analysis/versions/4
(5842, 2)


,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,2
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",0
2,"For the last quarter of 2010 , Componenta 's n...",2
3,According to the Finnish-Russian Chamber of Co...,1
4,The Swedish buyout firm has sold its remaining...,1


In [21]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
text_tokens = [str(text).lower() for text in df["Sentence"]]
encoding = tokenizer(text_tokens, return_tensors="pt", padding=True, truncation=True)
tokens = encoding["input_ids"]
attention_mask = encoding["attention_mask"]
labels = torch.tensor(df['Sentiment'], dtype=torch.long)

train_ds = TensorDataset(tokens, attention_mask, labels)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

In [25]:
num_epochs = 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5, weight_decay=5e-4)
model.train()
for epoch in range(num_epochs):
    loss_batch = 0.0
    accuracy_batch = 0.0
    count = 0

    for X, mask, y in tqdm(train_loader):
       X, mask, y = X.to(device), mask.to(device), y.to(device)
       optimizer.zero_grad()

       output = model(input_ids=X, attention_mask=mask, labels=y)
       logits = output.logits
       loss = output.loss
       loss.backward()
       optimizer.step()

       loss_batch += loss.item() * X.size(0)
       accuracy_batch += (logits.argmax(dim=1) == y).sum().item()
       count += X.size(0)

    print(f"Epoch {epoch}:")
    print(f"\tLoss: {loss_batch/count:.4f}, Accuracy: {accuracy_batch/count:.2%}")

100%|██████████| 92/92 [01:04<00:00,  1.43it/s]


Epoch 0:
	Loss: 0.7478, Accuracy: 66.28%


100%|██████████| 92/92 [01:04<00:00,  1.43it/s]


Epoch 1:
	Loss: 0.4220, Accuracy: 80.55%


100%|██████████| 92/92 [01:04<00:00,  1.43it/s]

Epoch 2:
	Loss: 0.3084, Accuracy: 85.24%
